In [6]:
# ================================================================
# ADVANCED TIME SERIES FORECASTING WITH ATTENTION-BASED RNN
# Multivariate dataset generation + SARIMAX baseline +
# Luong Attention BiLSTM forecasting model (PyTorch)
# ================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

# ============================================================
# 1. GENERATE SYNTHETIC MULTIVARIATE DATASET (5 FEATURES)
# ============================================================

def generate_dataset(n=2000):
    t = np.arange(n)

    # Feature 1 – Sine wave
    f1 = np.sin(0.02 * t) + 0.1 * np.random.randn(n)

    # Feature 2 – Correlated sine wave
    f2 = 0.7 * f1 + 0.3 * np.sin(0.05 * t) + 0.1 * np.random.randn(n)

    # Feature 3 – Trend + noise
    f3 = 0.001 * t + 0.3 * np.random.randn(n)

    # Feature 4 – Seasonal pattern
    f4 = np.sin(0.005 * t) * 0.5 + 0.1 * np.random.randn(n)

    # Feature 5 – Combination
    f5 = 0.4 * f1 + 0.4 * f3 + 0.2 * f4 + 0.2 * np.random.randn(n)

    df = pd.DataFrame({
        "f1": f1, "f2": f2, "f3": f3, "f4": f4, "f5": f5
    })

    df.to_csv("/content/synthetic_timeseries.csv", index=False)
    return df


df = generate_dataset()
print("Dataset saved as synthetic_timeseries.csv")
print(df.head())

# ============================================================
# 2. PREPROCESSING
# ============================================================

scaler = StandardScaler()
scaled = scaler.fit_transform(df)

SEQ_LEN = 50

def create_sequences(data, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len, 0])  # Predict feature f1
    return np.array(X), np.array(y)


X, y = create_sequences(scaled)
train_size = int(0.8 * len(X))

X_train = torch.tensor(X[:train_size], dtype=torch.float32)
y_train = torch.tensor(y[:train_size], dtype=torch.float32)

X_test = torch.tensor(X[train_size:], dtype=torch.float32)
y_test = torch.tensor(y[train_size:], dtype=torch.float32)

# ============================================================
# 3. DATASET CLASS
# ============================================================

class TSData(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(TSData(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TSData(X_test, y_test), batch_size=32)

# ============================================================
# 4. IMPLEMENT ATTENTION LAYER (LUONG)
# ============================================================

class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch, hidden)
        # encoder_outputs: (batch, seq, hidden)
        scores = torch.bmm(encoder_outputs, hidden.unsqueeze(2)).squeeze(2)
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context.squeeze(1), attn_weights


# ============================================================
# 5. ATTENTION-BASED BiLSTM FORECASTER
# ============================================================

class AttentionModel(nn.Module):

    def __init__(self, input_dim=5, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size, batch_first=True, bidirectional=False)
        self.attention = LuongAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h, c) = self.lstm(x)
        h = h[-1]  # last hidden state
        context, attn_weights = self.attention(h, out)
        y = self.fc(context)
        return y, attn_weights


model = AttentionModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ============================================================
# 6. TRAINING LOOP
# ============================================================

for epoch in range(10):
    model.train()
    losses = []
    for Xb, yb in train_loader:
        pred, _ = model(Xb)
        loss = criterion(pred.squeeze(), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Epoch {epoch+1}, Loss = {np.mean(losses):.4f}")

# ============================================================
# 7. EVALUATION + ATTENTION EXTRACTION
# ============================================================

model.eval()
preds, trues, attns = [], [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        pred, attn = model(Xb)
        preds.extend(pred.squeeze().numpy())
        trues.extend(yb.numpy())
        attns.append(attn.numpy())  # store attention weights

rmse = np.sqrt(mean_squared_error(trues, preds))
mae = mean_absolute_error(trues, preds)
mape = np.mean(np.abs((np.array(trues) - np.array(preds)) / np.array(trues))) * 100

print("\n==== ATTENTION MODEL PERFORMANCE ====")
print("RMSE:", rmse)
print("MAE :", mae)
print("MAPE:", mape, "\n")


# ============================================================
# 8. BASELINE MODEL (SARIMAX) — UPDATED FOR STABILITY + SCALE
# ============================================================
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning as SMConvergenceWarning

# Use the same scaler used earlier: compute scaled f1 series so comparison is apples-to-apples.
# scaled is numpy array returned by scaler.fit_transform(df) earlier.
f1_scaled = scaled[:, 0]
series_scaled = pd.Series(f1_scaled, name="f1_scaled")

# Keep the same train/test split indexing you used for sequences
# train_size was computed earlier as int(0.8 * len(X))
train_baseline = series_scaled[: train_size + SEQ_LEN]
test_baseline = series_scaled[train_size + SEQ_LEN :]

# Build SARIMAX with safer options
sarimax = sm.tsa.statespace.SARIMAX(
    train_baseline,
    order=(2, 0, 2),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

# Try multiple optimizers until one fits without throwing a hard error.
optimizers = ["lbfgs", "powell", "nm", "bfgs", "cg"]
res = None
last_exc = None

for method in optimizers:
    try:
        with warnings.catch_warnings():
            # statsmodels emits ConvergenceWarning; we suppress to avoid noisy tracebacks here.
            warnings.simplefilter("ignore", SMConvergenceWarning)
            res_try = sarimax.fit(method=method, maxiter=1000, disp=False)
        # If fit succeeded and returned results object, accept it
        if hasattr(res_try, "params"):
            res = res_try
            used_method = method
            break
    except Exception as e:
        last_exc = e
        # try next optimizer

if res is None:
    # If all optimizers failed, raise the last exception so user can inspect it.
    raise RuntimeError("SARIMAX fit failed with all attempted optimizers.") from last_exc

# Forecast on the scaled test range
forecast_scaled = res.forecast(steps=len(test_baseline))

# Compute metrics in scaled space (comparable to the PyTorch model which was trained on scaled data)
rmse_b_scaled = np.sqrt(mean_squared_error(test_baseline, forecast_scaled))
mae_b_scaled = mean_absolute_error(test_baseline, forecast_scaled)

# Convert scaled forecasts back to original scale for interpretability
# scaler.mean_ and scaler.scale_ correspond to all features; feature 0 is f1
mean_f1 = scaler.mean_[0]
scale_f1 = scaler.scale_[0]

test_baseline_orig = test_baseline * scale_f1 + mean_f1
forecast_orig = forecast_scaled * scale_f1 + mean_f1

rmse_b_orig = np.sqrt(mean_squared_error(test_baseline_orig, forecast_orig))
mae_b_orig = mean_absolute_error(test_baseline_orig, forecast_orig)

print("==== BASELINE SARIMAX PERFORMANCE (SCALED SPACE) ====")
print(f"Optimizer used: {used_method}")
print("RMSE (scaled):", rmse_b_scaled)
print("MAE  (scaled):", mae_b_scaled)

print("\n==== BASELINE SARIMAX PERFORMANCE (ORIGINAL SCALE) ====")
print("RMSE (orig)  :", rmse_b_orig)
print("MAE  (orig)  :", mae_b_orig)

Dataset saved as synthetic_timeseries.csv
         f1        f2        f3        f4        f5
0  0.040139  0.037128 -0.071143  0.030189  0.059888
1 -0.131733 -0.141363 -0.349273  0.133321  0.003311
2  0.198335  0.142501  0.782622 -0.038551  0.723316
3  0.038642 -0.061689 -0.250335 -0.053181  0.134899
4  0.225022  0.152791  0.285038 -0.106502  0.255128
Epoch 1, Loss = 0.4630
Epoch 2, Loss = 0.1999
Epoch 3, Loss = 0.1707
Epoch 4, Loss = 0.0957
Epoch 5, Loss = 0.0462
Epoch 6, Loss = 0.0358
Epoch 7, Loss = 0.0304
Epoch 8, Loss = 0.0289
Epoch 9, Loss = 0.0270
Epoch 10, Loss = 0.0258

==== ATTENTION MODEL PERFORMANCE ====
RMSE: 0.1874216113199783
MAE : 0.14210791086825805
MAPE: 54.349224 

==== BASELINE SARIMAX PERFORMANCE (SCALED SPACE) ====
Optimizer used: lbfgs
RMSE (scaled): 0.15564264145992937
MAE  (scaled): 0.12274884720988764

==== BASELINE SARIMAX PERFORMANCE (ORIGINAL SCALE) ====
RMSE (orig)  : 0.11168132283927523
MAE  (orig)  : 0.08807839230180171
